In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi
!pip install ultralytics

In [ ]:
import os
import cv2
import yaml
import shutil
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objs as go
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from ultralytics import YOLO               # <-- YOLOv8
from mlxtend.plotting import plot_confusion_matrix
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Path to the images and masks
IMAGE_PATH = '/content/drive/MyDrive/Kaliber AI/Project 2/dataset/lgg-mri-segmentation/kaggle_3m'

# YOLOv8 works best with 640x640. Change from 256.
IMAGE_SIZE = (640, 640)   # fix: tuple instead of integer
YOLO_IMGSZ = 640          # used in model.train() and model.predict()

# Epochs
EPOCHS = 5

# Batch size
BATCH_SIZE = 30

# Path where we'll build the YOLO dataset folder
YOLO_DATASET_PATH = '/content/drive/MyDrive/Kaliber AI/Project 2/dataset/lgg-mri-segmentation/yolo_dataset'

In [ ]:
paths = []

# Use os.walk to find all .tif files within the IMAGE_PATH and its subdirectories
for root, _, files in os.walk(IMAGE_PATH):
    for file in files:
        if file.endswith('.tif'):
            paths.append(os.path.join(root, file))

len(paths), paths[:20:5]

In [ ]:
# Making a dataframe from the list "paths" by separating Images and Masks, extracting IDs, and diagnoses.
'''def data_frame(data):
    # Storing only paths that don't end with 'mask.tiff'
    images = list(filter(lambda x: not x.endswith('mask.tif'), data))
    # Sorting images based on the number of each MRI.
    images.sort(key=lambda x: int(x.rsplit('_', 3)[-1][:-4]))
    # Sorting by the patient IDs (each patient has more than 1 MRIs)
    images.sort(key=lambda x: int(x.rsplit('_', 3)[-2]))

    # Storing the image IDs
    IDs = list(map(lambda x: x.rsplit('/', 3)[-1][:-4], images))

    # Storing only paths that end with 'mask.tiff'
    masks = list(filter(lambda x: x.endswith('mask.tif'), data))
    # Sorting masks based on the number of each MRI.
    masks.sort(key=lambda x: int(x.rsplit('_', 3)[-2]))
    # Sorting by the patient IDs (each patient has more than 1 MRIs)
    masks.sort(key=lambda x: int(x.rsplit('_', 3)[-3]))

    # Opens the images
    pixels = lambda x: Image.open(x)
    # Selects the largest pixel
    largest_pixel = lambda y: np.max(pixels(y))
    # Determines if the mask contains an abnormality or not (+ or -)
    # Remember that a negative image's mask is just an entirely black image.
    diagnotic_function = lambda z: 1 if largest_pixel(z) > 0 else 0
    # Storing the diagnosis corresponding to each image
    diagnoses = list(map(lambda x: diagnotic_function(x), masks))

    # Making the dataframe
    DataFrame = pd.DataFrame({'ID': IDs, 'Image': images, 'Mask': masks, 'Diagnosis': diagnoses})

    # Dividing the indexes into train, test, and validation
    train_index, val_index = train_test_split(DataFrame.index.values.tolist(), test_size=0.25, random_state=42)
    val_index, test_index = train_test_split(val_index, test_size=0.15, random_state=42)

    # Making train, test, and validation dataframes
    train_df, val_df, test_df = DataFrame.iloc[train_index], DataFrame.iloc[val_index], DataFrame.iloc[test_index]

    return train_df, val_df, test_df

# Making the dataframes
train_df, val_df, test_df = data_frame(paths)

print(len(train_df), len(val_df), len(test_df))

train_df.head()
val_df.head()
test_df.head()'''





'''def data_frame(data):
    # Separate images and masks
    images = sorted([x for x in data if not x.endswith('mask.tif')])
    masks  = sorted([x for x in data if x.endswith('mask.tif')])

    # Match each image to its corresponding mask by base name
    # e.g. "TCGA_CS_4941_19960909_1.tif" should match "TCGA_CS_4941_19960909_1_mask.tif"
    matched_images = []
    matched_masks  = []

    for img_path in images:
        # Build the expected mask filename from the image filename
        expected_mask = img_path.replace('.tif', '_mask.tif')
        if expected_mask in masks:
            matched_images.append(img_path)
            matched_masks.append(expected_mask)

    # Store IDs
    IDs = [x.rsplit('/', 1)[-1][:-4] for x in matched_images]

    # Determine diagnosis from mask
    pixels          = lambda x: Image.open(x)
    largest_pixel   = lambda y: np.max(pixels(y))
    diagnotic_function = lambda z: 1 if largest_pixel(z) > 0 else 0
    diagnoses = list(map(diagnotic_function, matched_masks))

    # Sanity check
    print(f"Matched images: {len(matched_images)}, Masks: {len(matched_masks)}, IDs: {len(IDs)}, Diagnoses: {len(diagnoses)}")

    # Build DataFrame
    DataFrame = pd.DataFrame({
        'ID': IDs,
        'Image': matched_images,
        'Mask': matched_masks,
        'Diagnosis': diagnoses
    })

    # Split into train, val, test
    train_index, val_index = train_test_split(DataFrame.index.values.tolist(), test_size=0.25, random_state=42)
    val_index, test_index  = train_test_split(val_index, test_size=0.15, random_state=42)

    train_df = DataFrame.iloc[train_index]
    val_df   = DataFrame.iloc[val_index]
    test_df  = DataFrame.iloc[test_index]

    return train_df, val_df, test_df

# Making the dataframes
train_df, val_df, test_df = data_frame(paths)
print(len(train_df), len(val_df), len(test_df))'''



def data_frame(data):
    images = sorted([x for x in data if not x.endswith('mask.tif')])
    masks  = sorted([x for x in data if x.endswith('mask.tif')])

    # Match each image to its mask directly by filename
    matched_images = []
    matched_masks  = []
    for img_path in images:
        expected_mask = img_path.replace('.tif', '_mask.tif')
        if expected_mask in set(masks):  # set() makes lookup faster
            matched_images.append(img_path)
            matched_masks.append(expected_mask)

    IDs = [x.rsplit('/', 1)[-1][:-4] for x in matched_images]

    # Determine diagnosis with a progress bar so you can see it working
    diagnoses = []
    for mask_path in tqdm(matched_masks, desc='Reading masks'):
        img_array = np.array(Image.open(mask_path))
        diagnoses.append(1 if np.max(img_array) > 0 else 0)

    DataFrame = pd.DataFrame({
        'ID': IDs,
        'Image': matched_images,
        'Mask': matched_masks,
        'Diagnosis': diagnoses
    })

    train_index, val_index = train_test_split(DataFrame.index.values.tolist(), test_size=0.25, random_state=42)
    val_index, test_index  = train_test_split(val_index, test_size=0.15, random_state=42)

    return DataFrame.iloc[train_index], DataFrame.iloc[val_index], DataFrame.iloc[test_index]

train_df, val_df, test_df = data_frame(paths)
print(len(train_df), len(val_df), len(test_df))

In [ ]:
def plot_images_and_masks(images, masks, predictions=None, IoU_list=None):
    num_samples = len(images)
    # Checking whether the function has been given the prediction array or not
    num_rows = 2 if type(predictions) == type(None) else 3

    # Defining figure
    fig, axes = plt.subplots(num_rows, num_samples,
                             figsize=(num_samples*5, num_samples+(num_rows*2)), dpi=200)

    for i in range(num_samples):
        # Plotting image
        axes[0, i].imshow(Image.open(images[i]), cmap='gray')
        axes[0, i].set_title('Image', fontsize=20, fontweight='bold')
        axes[0, i].axis('off')

        # Plotting mask
        axes[1, i].imshow(Image.open(masks[i]), cmap='gray')
        axes[1, i].set_title('Mask', fontsize=20, fontweight='bold')
        axes[1, i].axis('off')

        # Plotting prediction
        if type(predictions) != type(None):
            axes[2, i].imshow(predictions[i], cmap='gray')
            axes[2, i].set_title(f'Prediction | IoU: {round(float(IoU_list[i]), 3)}',
                                 fontsize=19, fontweight='bold')
            axes[2, i].axis('off')

    # Adding title
    plt.suptitle('Images and Masks', fontsize=30, fontweight='bold')

    # Showing the figure
    plt.show()

plot_images_and_masks(train_df[train_df['Diagnosis'] == 1]['Image'].values[:10],
                      train_df[train_df['Diagnosis'] == 1]['Mask'].values[:10])

In [ ]:
plot_images_and_masks(train_df[train_df['Diagnosis'] == 0]['Image'].values[:10],
                      train_df[train_df['Diagnosis'] == 0]['Mask'].values[:10])

In [ ]:
def plot_class_distribution(train_df, val_df, test_df):
    # Counting the class distribution for every dataframe.
    class_distribution_df1 = train_df['Diagnosis'].value_counts()
    class_distribution_df2 = val_df['Diagnosis'].value_counts()
    class_distribution_df3 = test_df['Diagnosis'].value_counts()

    colors = ['#0504AA', '#ED0101']

    # Defining figure
    fig = go.Figure()

    # Adding bars for each dataframe and each class
    for class_label in class_distribution_df1.index:
        # Getting the name of each class (+ or -)
        class_name = 'Positive' if class_label == 1 else 'Negative'

        # Creating stacked bar for each class
        fig.add_trace(go.Bar(
            x=['Training', 'Validation', 'Test'],
            y=[class_distribution_df1.get(class_label, 0),
               class_distribution_df2.get(class_label, 0),
               class_distribution_df3.get(class_label, 0)],

            name=f'{class_name}',
            marker=dict(color=colors[class_label]),
            opacity=0.75,
            width=0.3
        ))

    # Updating layout
    fig.update_layout(
        height=700,
        width=800,
        title_text="Class Distribution",
        title_font=dict(size=25, family='Balto'),
        title_x=0.5,
        title_y=0.98,
        xaxis=dict(title='Dataframes', title_font=dict(family='Balto', size=19), tickfont=dict(family='Balto', size=19)),
        yaxis=dict(title='Count', title_font=dict(family='Balto', size=19), tickfont=dict(family='Balto', size=19)),
        margin=dict(l=60, r=20, t=50, b=40),
        legend=dict(x=0.73, y=0.98, traceorder='normal', orientation='h', font=dict(family='Balto')),  # Placing legend inside
        barmode='stack'  # Setting barmode to 'stack' to stack the bars
                )
    # Showing the figure
    fig.show()

plot_class_distribution(train_df, val_df, test_df)

In [ ]:
# Processes the image
def decode_and_resize_image(img_path):
    # Reading '.tiff' format image
    img = tf.io.read_file(img_path)
    with tf.io.gfile.GFile(img_path, 'rb') as f:
        img = Image.open(f)
        img = np.array(img)
    img = tf.convert_to_tensor(img, dtype=tf.float32)
    img = tf.image.resize(img, IMAGE_SIZE, preserve_aspect_ratio=True)

    # Normalizing the image to the range [0, 1]
    img = img / 255.0

    # Pbar updating, started at the data setup cell.
    try:
        pbar.update(1)
    except:
        pass

    return img

# Processes the mask
def decode_and_resize_mask(mask_path):
    # Reading '.tiff' format masks
    mask = tf.io.read_file(mask_path)
    with tf.io.gfile.GFile(mask_path, 'rb') as f:
        mask = Image.open(f)
        mask = np.array(mask)
    mask = tf.convert_to_tensor(mask, dtype=tf.float32)
    mask = np.expand_dims(mask, axis=-1)
    mask = tf.image.resize(mask, IMAGE_SIZE, method='nearest', preserve_aspect_ratio=True)
    grayscale_mask = tf.reduce_mean(mask, axis=-1, keepdims=True)

    # Normalizing the mask to the range [0, 1]
    grayscale_mask = grayscale_mask / 255.0

    # Pbar updating, started at the data setup cell.
    try:
        pbar.update(1)
    except:
        pass

    return grayscale_mask

In [ ]:
def processed_input(img, mask):
    # Processed images: (None, 256, 256, 3), Processed masks: (None, 256, 256, 1)
    # This portion could be used for Data Augmentation in future use cases
    return img, mask

# Prepares the dataset
def make_dataset(images, masks):
    dataset = tf.data.Dataset.from_tensor_slices((list(map(lambda x: decode_and_resize_image(x), images)),
                                                 list(map(lambda x: decode_and_resize_mask(x), masks))))
    # Shuffle dataset
    dataset = dataset.shuffle(BATCH_SIZE * 8)
    # Map dataset
    dataset = dataset.map(processed_input, num_parallel_calls=tf.data.AUTOTUNE)
    # While the current batch of data is being processed, prefetching the next batch based on available resources.
    dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return dataset

# Defining  a Pbar to keep an eye on the procedure.
pbar = tqdm(total=(len(train_df)+len(val_df))*2, position=0, leave=True, colour='green')

# Making the datasets by providing the lists of corresponding masks and images.
#train_dataset = make_dataset(list(train_df['Image'].values), list(train_df['Mask'].values))
#validation_dataset = make_dataset(list(val_df['Image'].values), list(val_df['Mask'].values))

pbar.close()

In [ ]:
def mask_to_yolo_polygon(mask_path, image_size=640):
    """
    Converts a binary mask image into a YOLO segmentation polygon string.
    Returns None if no tumor found (negative/all-black mask).
    """
    mask = np.array(Image.open(mask_path).convert('L').resize((image_size, image_size)))
    mask = (mask > 0).astype(np.uint8)  # Binarize

    # Find contours (outlines) of the white region in the mask
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None  # No tumor — negative image

    # Take the largest contour (main tumor region)
    largest = max(contours, key=cv2.contourArea)

    # Flatten and normalize to [0, 1] range as YOLO requires
    points = largest.flatten().astype(float)
    points[0::2] /= image_size  # x coords
    points[1::2] /= image_size  # y coords

    # YOLO format: "class_id x1 y1 x2 y2 ..."
    polygon_str = '0 ' + ' '.join(map(str, points))
    return polygon_str


def build_yolo_dataset(df, split_name):
    """
    Copies images and creates label .txt files for a given split (train/val/test).
    """
    img_out = os.path.join(YOLO_DATASET_PATH, 'images', split_name)
    lbl_out = os.path.join(YOLO_DATASET_PATH, 'labels', split_name)
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'Building {split_name}'):
        img_id = row['ID']
        img_src = row['Image']
        mask_src = row['Mask']

        # Convert .tif image to .jpg and copy
        img = Image.open(img_src).convert('RGB').resize((YOLO_IMGSZ, YOLO_IMGSZ), Image.BILINEAR)
        img.save(os.path.join(img_out, f'{img_id}.jpg'))

        # Generate label file
        polygon = mask_to_yolo_polygon(mask_src, YOLO_IMGSZ)
        lbl_path = os.path.join(lbl_out, f'{img_id}.txt')

        if polygon:
            with open(lbl_path, 'w') as f:
                f.write(polygon)
        else:
            # Negative image: write empty label file (no tumor)
            open(lbl_path, 'w').close()

# Build all splits
build_yolo_dataset(train_df, 'train')
build_yolo_dataset(val_df,   'val')
build_yolo_dataset(test_df,  'test')

# Create the dataset.yaml file YOLOv8 needs
yaml_content = {
    'path': YOLO_DATASET_PATH,
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc': 1,               # number of classes (just "tumor")
    'names': ['tumor']
}
with open(os.path.join(YOLO_DATASET_PATH, 'dataset.yaml'), 'w') as f:
    yaml.dump(yaml_content, f)

print("✅ YOLO dataset ready!")

In [ ]:
# Load YOLOv8 segmentation model (pretrained on COCO dataset)
# 'yolov8n-seg.pt' = nano (fastest), you can use 'm', 'l', 'x' for more accuracy
model = YOLO('yolov8n-seg.pt')

print("✅ YOLOv8 segmentation model loaded!")
model.info()  # Prints model summary

In [ ]:
# Training the YOLOv8 model
results = model.train(
    data=os.path.join(YOLO_DATASET_PATH, 'dataset.yaml'),
    epochs=EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=BATCH_SIZE,
    patience=6,          # same as your EarlyStopping patience
    save=True,           # saves best weights automatically
    project='/content/drive/MyDrive/Kaliber AI/Project 2/dataset/lgg-mri-segmentation/yolo_runs',
    name='brain_tumor_seg',
    exist_ok=True
)

print("✅ Training complete!")
print(f"Best weights saved at: {results.save_dir}")

In [ ]:
def plot_history(results):
    # YOLOv8 saves training metrics to a CSV
    csv_path = os.path.join(results.save_dir, 'results.csv')
    df_results = pd.read_csv(csv_path)
    df_results.columns = df_results.columns.str.strip()  # Clean column names

    epochs = df_results['epoch'].tolist()
    train_loss = df_results['train/seg_loss'].tolist()
    val_loss   = df_results['val/seg_loss'].tolist()

    plots = go.Figure([
        go.Scatter(x=epochs, y=train_loss, name='Training Loss',
                   mode='lines+markers', marker=dict(color='#00008B'), line=dict(width=1)),
        go.Scatter(x=epochs, y=val_loss, name='Validation Loss',
                   mode='lines+markers', marker=dict(color="#004EFF"), line=dict(width=1))
    ])

    plots.update_layout(
        title_text="Training vs Validation",
        title_font=dict(size=25, family='Balto'),
        title_x=0.5,
        xaxis=dict(title='Epoch', title_font=dict(family='Balto', size=19)),
        yaxis=dict(title='Loss',  title_font=dict(family='Balto', size=19)),
        legend=dict(x=0.75, y=0.98, orientation='h', font=dict(family='Balto'))
    )
    plots.show()

plot_history(results)

In [ ]:
def IoU(y_true, y_pred):
    # This function stays exactly the same
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true + y_pred, axis=[1,2,3]) - intersection
    IoU = (intersection + 1e-25) / (union + 1e-25)
    return IoU


def predictions(df, threshold=0.5):
    # Load best trained weights
    best_model = YOLO(os.path.join('/content/drive/MyDrive/Kaliber AI/Project 2/dataset/lgg-mri-segmentation/yolo_runs/brain_tumor_seg/weights/best.pt'))

    pred_masks = []
    diagnoses  = []
    ids        = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = row['Image']

        # Run YOLOv8 prediction
        result = best_model.predict(source=img_path, imgsz=IMAGE_SIZE, conf=threshold, verbose=False)[0]

        # Build a blank mask
        mask_out = np.zeros((YOLO_IMGSZ, YOLO_IMGSZ, 1), dtype=np.float32)

        if result.masks is not None:
            # Get the predicted mask and resize to IMAGE_SIZE
            raw_mask = result.masks.data[0].cpu().numpy()
            raw_mask = cv2.resize(raw_mask, (YOLO_IMGSZ, YOLO_IMGSZ))
            mask_out[:, :, 0] = raw_mask
            diagnoses.append(1)  # Tumor found
        else:
            diagnoses.append(0)  # No tumor

        pred_masks.append(mask_out)
        ids.append(row['ID'])

    # Get ground truth masks
    y_true = np.array([decode_and_resize_mask(row['Mask']) for _, row in df.iterrows()])
    y_pred = np.array(pred_masks)

    # Calculate IoU
    IoUs = IoU(y_true, y_pred).numpy()
    IoUs = [round(float(i), 3) for i in IoUs]

    prediction_df = pd.DataFrame({
        'ID': df['ID'].values,
        'Actual Diagnosis': df['Diagnosis'].values,
        'Predicted Diagnosis': diagnoses,
        'IoU': IoUs
    })

    prediction_dict = {i: tf.constant(m) for i, m in zip(ids, pred_masks)}
    return prediction_dict, prediction_df


# Run predictions
prediction, prediction_df = predictions(test_df)
print(len(prediction), list(prediction.values())[0].shape)
prediction_df.head()

In [ ]:
# Get IDs where actual diagnosis is positive (tumor present)
positive_ids = prediction_df[prediction_df['Actual Diagnosis'] == 1]['ID'].values

# Get IDs where actual diagnosis is negative (no tumor)
negative_ids = prediction_df[prediction_df['Actual Diagnosis'] == 0]['ID'].values

print(f"Positive cases: {len(positive_ids)}")
print(f"Negative cases: {len(negative_ids)}")

In [ ]:
plot_images_and_masks([test_df.loc[test_df['ID'] == ID, 'Image'].values[0] for ID in positive_ids[:8]],
                      [test_df.loc[test_df['ID'] == ID, 'Mask'].values[0] for ID in positive_ids[:8]],
                      [prediction[ID].numpy() for ID in positive_ids[:8]],
                     [prediction_df.loc[prediction_df['ID'] == ID, 'IoU'].values[0] for ID in positive_ids[:8]])

In [ ]:
plot_images_and_masks([test_df.loc[test_df['ID'] == ID, 'Image'].values[0] for ID in positive_ids[8:16]],
                      [test_df.loc[test_df['ID'] == ID, 'Mask'].values[0] for ID in positive_ids[8:16]],
                      [prediction[ID].numpy() for ID in positive_ids[8:16]],
                     [prediction_df.loc[prediction_df['ID'] == ID, 'IoU'].values[0] for ID in positive_ids[8:16]])

In [ ]:
plot_images_and_masks([test_df.loc[test_df['ID'] == ID, 'Image'].values[0] for ID in positive_ids[16:]],
                      [test_df.loc[test_df['ID'] == ID, 'Mask'].values[0] for ID in positive_ids[16:]],
                      [prediction[ID].numpy() for ID in positive_ids[16:]],
                     [prediction_df.loc[prediction_df['ID'] == ID, 'IoU'].values[0] for ID in positive_ids[16:]])

In [ ]:
plot_images_and_masks([test_df.loc[test_df['ID'] == ID, 'Image'].values[0] for ID in negative_ids[:8]],
                      [test_df.loc[test_df['ID'] == ID, 'Mask'].values[0] for ID in negative_ids[:8]],
                      [prediction[ID].numpy() for ID in negative_ids[:8]],
                     [prediction_df.loc[prediction_df['ID'] == ID, 'IoU'].values[0] for ID in negative_ids[:8]])

In [ ]:
plot_images_and_masks([test_df.loc[test_df['ID'] == ID, 'Image'].values[0] for ID in negative_ids[8:16]],
                      [test_df.loc[test_df['ID'] == ID, 'Mask'].values[0] for ID in negative_ids[8:16]],
                      [prediction[ID].numpy() for ID in negative_ids[8:16]],
                     [prediction_df.loc[prediction_df['ID'] == ID, 'IoU'].values[0] for ID in negative_ids[8:16]])

In [ ]:
plot_images_and_masks([test_df.loc[test_df['ID'] == ID, 'Image'].values[0] for ID in negative_ids[16:]],
                      [test_df.loc[test_df['ID'] == ID, 'Mask'].values[0] for ID in negative_ids[16:]],
                      [prediction[ID].numpy() for ID in negative_ids[16:]],
                     [prediction_df.loc[prediction_df['ID'] == ID, 'IoU'].values[0] for ID in negative_ids[16:]])

In [ ]:
# Making predictions
prediction, prediction_df = predictions(pd.concat([test_df, val_df], ignore_index=True))

print(len(prediction), list(prediction.values())[0].shape)

prediction_df.head()

In [ ]:
prediction_df.describe()

In [ ]:
def plot_class_distribution(df):
    # Counting the actual and predicted class distributions (+, -).
    class_distribution_df1 = df['Actual Diagnosis'].value_counts()
    class_distribution_df2 = df['Predicted Diagnosis'].value_counts()
    mean_IoU = round(df['IoU'].mean(), 3)

    colors = ['#0504AA', '#ED0101']

    # Defining figure
    fig = go.Figure()

    # Adding bars for each dataframe and each class
    for class_label in class_distribution_df1.index:
        # Getting the name of each class (+ or -)
        class_name = 'Positive' if class_label == 1 else 'Negative'

        # Creating stacked bar for each class
        fig.add_trace(go.Bar(
            x=['Actual', 'Predicted'],
            y=[class_distribution_df1.get(class_label, 0),
               class_distribution_df2.get(class_label, 0)],

            name=f'{class_name}',
            marker=dict(color=colors[class_label]),
            opacity=0.75,
            width=0.25
        ))

    # Updating layout
    fig.update_layout(
        height=700,
        width=800,
        title_text=f"Class Distribution | IoU {round(float(mean_IoU), 4)}",
        title_font=dict(size=25, family='Balto'),
        title_x=0.5,
        title_y=0.98,
        xaxis=dict(title='Columns', title_font=dict(family='Balto', size=19), tickfont=dict(family='Balto', size=19)),
        yaxis=dict(title='Count', title_font=dict(family='Balto', size=19), tickfont=dict(family='Balto', size=19)),
        margin=dict(l=60, r=20, t=50, b=40),
        legend=dict(x=0.38, y=0.98, traceorder='normal', orientation='h', font=dict(family='Balto')),  # Placing legend inside
        barmode='stack'  # Setting barmode to 'stack' to stack the bars
                )
    # Showing the figure
    fig.show()

plot_class_distribution(prediction_df)

In [ ]:
# Plots Confusion Matrix
def Confusion_Matrix(df):
    confusionMatrix = confusion_matrix(df['Actual Diagnosis'], df['Predicted Diagnosis'])

    # Defining class names
    class_names = ['Negative', 'Positive']

    # Normalizing confusion matrix
    confusionMatrixNormalized = confusionMatrix.astype('float') / confusionMatrix.sum(axis=1)[:, np.newaxis]

    # Creating heatmap trace for confusion matrix
    heatmap_trace = go.Heatmap(z=confusionMatrixNormalized,
                               x=class_names,
                               y=class_names,
                               colorscale='Viridis',
                               colorbar=dict(title='Proportion', title_font=dict(family='Balto'),
                                             tickfont=dict(family='Balto'), tickformat='.2f'))

    # Creating text annotations for each cell in the heatmap
    annotations = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            annotations.append(dict(text=str(confusionMatrix[i, j]),
                                    x=class_names[j],
                                    y=class_names[i],
                                    showarrow=False,
                                    font=dict(color='black', family='Balto', size=25)))

    # Creating layout
    layout = go.Layout(height=700,
                       width=800,
                       title='Confusion Matrix',
                       title_font=dict(size=25, family='Balto'),
                       title_x=0.5,
                       title_y=0.95,
                       xaxis=dict(title='Predicted label', title_font=dict(family='Balto', size=19), tickfont=dict(family='Balto', size=19)),
                       yaxis=dict(title='True label', title_font=dict(family='Balto', size=19), tickfont=dict(family='Balto', size=19)),
                       annotations=annotations)

    # Creating figure
    fig = go.Figure(data=[heatmap_trace], layout=layout)

    # Showing the plot
    fig.show()

Confusion_Matrix(prediction_df)

In [ ]:
# Rounds the number of Classification Report output
def round_dict(d, decimals=2):
    if isinstance(d, dict):
        return {key: round_dict(value, decimals) for key, value in d.items()}
    else:
        return round(d, decimals)

# Plots Classification Report
def Classification_Report(df):
    clf_report = classification_report(df['Actual Diagnosis'],
                                       df['Predicted Diagnosis'],
                                       labels=np.arange(2),
                                       target_names=['Negative', 'Positive'],
                                       output_dict=True)

    rounded_data = round_dict(clf_report, decimals=2)

    # Converting classification report data to dataframe
    df_clf_report = pd.DataFrame.from_dict(rounded_data)

    # Transposing dataframe for proper visualization
    df_clf_report = df_clf_report.T

    # Reversing the order of rows in the dataframe
    df_clf_report = df_clf_report.iloc[::-1]

    # Creating heatmap trace for classification report
    heatmap_trace = go.Heatmap(z=df_clf_report.values[:, :-1],  # Excluding 'support' column
                               x=df_clf_report.columns[:-1],  # Excluding 'support' column
                               y=df_clf_report.index,
                               colorscale='Viridis')

    # Creating text annotations for each cell in the heatmap
    annotations = []
    for i in range(len(df_clf_report.index)):
        for j in range(len(df_clf_report.columns) - 1):
            annotations.append(dict(text=str(df_clf_report.values[i, j]),
                                    x=df_clf_report.columns[j],
                                    y=df_clf_report.index[i],
                                    showarrow=False,
                                    font=dict(color='black', family='Balto', size=22)))

    # Creating layout
    layout = go.Layout(height=800,
                       width=800,
                       title='Classification Report',
                       title_font=dict(size=25, family='Balto'),
                       title_x=0.5,
                       title_y=0.95,
                       xaxis=dict(title='Metrics', title_font=dict(family='Balto', size=19), tickfont=dict(family='Balto', size=19)),
                       yaxis=dict(title='Class', title_font=dict(family='Balto', size=19), tickfont=dict(family='Balto', size=19)),
                       annotations=annotations)

    # Creating figure
    fig = go.Figure(data=[heatmap_trace], layout=layout)

    # Showing the plot
    fig.show()

Classification_Report(prediction_df)